In [3]:
import pandas as pd
from IPython.display import display

df_state_path = "df_state.xlsx"
voyageurs_path = "voyageurs.xlsx"

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

df_state = pd.read_excel(df_state_path)
df_voyageurs = pd.read_excel(voyageurs_path)

# =========================
# CLÉ TRAIN UNIQUEMENT
# =========================
df_state_cmp = df_state.copy()
df_voy_cmp = df_voyageurs.copy()

df_state_cmp["cle_train"] = (
    df_state_cmp["train_number"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

df_voy_cmp["cle_train"] = (
    df_voy_cmp["numero_train"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

trains_state = df_state_cmp[["cle_train"]].drop_duplicates()
trains_voy = df_voy_cmp[["cle_train"]].drop_duplicates()

# =========================
# TRAINS DANS df_state MAIS PAS DANS voyageurs
# =========================
trains_absents_voyageurs = trains_state.merge(
    trains_voy,
    on="cle_train",
    how="left",
    indicator=True
)

trains_absents_voyageurs = trains_absents_voyageurs[
    trains_absents_voyageurs["_merge"] == "left_only"
].drop(columns="_merge")

# =========================
# TRAINS DANS voyageurs MAIS PAS DANS df_state
# =========================
trains_absents_df_state = trains_voy.merge(
    trains_state,
    on="cle_train",
    how="left",
    indicator=True
)

trains_absents_df_state = trains_absents_df_state[
    trains_absents_df_state["_merge"] == "left_only"
].drop(columns="_merge")

# =========================
# DETAILS PAR TRAIN
# =========================
df_trains_absents_voyageurs = (
    df_state_cmp[df_state_cmp["cle_train"].isin(trains_absents_voyageurs["cle_train"])]
    .groupby("train_number", as_index=False)
    .agg(
        nb_dessertes_df_state=("stop_name", "count"),
        gare_depart=("stop_name", "first"),
        terminus=("stop_name", "last"),
        statut=("train_status", "first"),
    )
    .sort_values("train_number")
)

df_trains_absents_df_state = (
    df_voy_cmp[df_voy_cmp["cle_train"].isin(trains_absents_df_state["cle_train"])]
    .groupby("numero_train", as_index=False)
    .agg(
        nb_dessertes_voyageurs=("des_lib_ext", "count"),
        gare_depart=("tra_lib_ext_gare_depart", "first"),
        terminus=("tra_lib_ext_gare_arrivee", "first"),
    )
    .sort_values("numero_train")
)

# =========================
# AFFICHAGE
# =========================
print("======================================================")
print(f"Trains présents dans df_state mais absents de voyageurs : {len(df_trains_absents_voyageurs)}")
print("======================================================")
display(df_trains_absents_voyageurs)

print("======================================================")
print(f"Trains présents dans voyageurs mais absents de df_state : {len(df_trains_absents_df_state)}")
print("======================================================")
display(df_trains_absents_df_state)

# =========================
# EXPORT
# =========================
df_trains_absents_voyageurs.to_excel("trains_df_state_absents_voyageurs.xlsx", index=False)
df_trains_absents_df_state.to_excel("trains_voyageurs_absents_df_state.xlsx", index=False)

print("Fichiers exportés :")
print("- trains_df_state_absents_voyageurs.xlsx")
print("- trains_voyageurs_absents_df_state.xlsx")

TypeError: '<' not supported between instances of 'str' and 'int'

In [2]:
import pandas as pd
from pathlib import Path

# =========================
# CHEMINS FICHIERS
# =========================
BASE_DIR = Path.cwd()

DF_STATE_FILE = BASE_DIR / "df_state.xlsx"
VOYAGEURS_FILE = BASE_DIR / "voyageurs.xlsx"

# =========================
# LECTURE
# =========================
df_state = pd.read_excel(DF_STATE_FILE)
df_voy = pd.read_excel(VOYAGEURS_FILE)

# =========================
# VERIFICATION COLONNES
# =========================
if "stop_name" not in df_state.columns:
    raise ValueError("Colonne 'stop_name' absente dans df_state.xlsx")

if "des_lib_ext" not in df_voy.columns:
    raise ValueError("Colonne 'des_lib_ext' absente dans voyageurs.xlsx")

# =========================
# EXTRACTION UNIQUE
# =========================
gares_state = (
    df_state["stop_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .sort_values()
    .unique()
)

gares_voy = (
    df_voy["des_lib_ext"]
    .dropna()
    .astype(str)
    .str.strip()
    .sort_values()
    .unique()
)

# =========================
# AFFICHAGE CONSOLE
# =========================
print("\n==============================")
print("GARES EXISTANTES DANS DF_STATE")
print("==============================\n")

for gare in gares_state:
    print(gare)

print(f"\nTOTAL DF_STATE : {len(gares_state)} gares uniques")

print("\n==============================")
print("GARES EXISTANTES DANS VOYAGEURS")
print("==============================\n")

for gare in gares_voy:
    print(gare)

print(f"\nTOTAL VOYAGEURS : {len(gares_voy)} gares uniques")

# =========================
# EXPORT EXCEL COMPARATIF
# =========================
max_len = max(len(gares_state), len(gares_voy))

df_export = pd.DataFrame({
    "gares_df_state": list(gares_state) + [None] * (max_len - len(gares_state)),
    "gares_voyageurs": list(gares_voy) + [None] * (max_len - len(gares_voy)),
})

output_file = BASE_DIR / "comparatif_gares_uniques.xlsx"
df_export.to_excel(output_file, index=False)

print(f"\nFichier exporté : {output_file}")


GARES EXISTANTES DANS DF_STATE

Aix-en-Provence TGV
Angers Saint-Laud
Angoulême
Antibes
Augsburg Hbf
Avignon TGV
Aéroport Charles de Gaulle 2 TGV
Baden-Baden
Bar-le-Duc
Beaune
Belfort - Montbéliard TGV
Berlin Hbf-Lehrter Bahnhof Nord
Berlin Suedkreuz
Berlin-Gesundbrunnen
Besançon Franche-Comté TGV
Bordeaux Saint-Jean
Bruxelles Midi
Cannes
Chalon-sur-Saône
Champagne-Ardenne TGV
Charleville-Mézières
Châlons-en-Champagne
Colmar
Dijon
Erfurt Hbf
Forbach
Francfort sur le Main
Freiburg (Breisgau) Hbf
Halle (Saale) Hbf
Kaiserslautern Hbf
Karlsruhe Hbf
Lahr (Schwarzw)
Le Mans
Les Arcs - Draguignan
Lille Europe
Lorraine TGV
Lunéville
Luxembourg
Lyon Part Dieu
Mannheim Hbf
Marne-la-Vallée Chessy
Marseille Saint-Charles
Massy TGV
Metz
Meuse TGV
Montbard
Montpellier Saint-Roch
Mulhouse
Munich
Mâcon
Nancy
Nantes
Nice-Ville
Nîmes Centre
Offenburg
Paris Est
Paris Gare de Lyon Hall 1 - 2
Poitiers
Reims
Remiremont
Rennes
Rethel
Ringsheim
Saint-Dié-des-Vosges
Saint-Pierre-des-Corps
Saint-Raphaël Valesc

In [4]:
import requests
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict

TRAIN_SEARCH = "9877"

URL = "https://proxy.transport.data.gouv.fr/resource/sncf-gtfs-rt-trip-updates"

feed = gtfs_realtime_pb2.FeedMessage()

response = requests.get(URL, timeout=20)
response.raise_for_status()
feed.ParseFromString(response.content)

found = False

for entity in feed.entity:
    if not entity.HasField("trip_update"):
        continue

    tu = entity.trip_update
    trip_id = tu.trip.trip_id

    if TRAIN_SEARCH not in trip_id:
        continue

    found = True

    print("=" * 120)
    print("ENTITY ID :", entity.id)
    print("TRIP ID   :", trip_id)
    print("START DATE:", tu.trip.start_date)
    print("TIMESTAMP :", tu.timestamp)
    print("=" * 120)

    for i, stu in enumerate(tu.stop_time_update, start=1):
        print(f"\n--- Stop update {i} ---")
        print("stop_sequence :", stu.stop_sequence)
        print("stop_id       :", stu.stop_id)
        print("relationship  :", stu.schedule_relationship)

        if stu.HasField("arrival"):
            print("arrival delay :", stu.arrival.delay if stu.arrival.HasField("delay") else None)
            print("arrival time  :", stu.arrival.time if stu.arrival.HasField("time") else None)

        if stu.HasField("departure"):
            print("departure delay :", stu.departure.delay if stu.departure.HasField("delay") else None)
            print("departure time  :", stu.departure.time if stu.departure.HasField("time") else None)

    print("\n\n--- ENTITÉ COMPLÈTE EN DICT ---")
    print(MessageToDict(entity))

if not found:
    print(f"Aucune entité trouvée pour le train {TRAIN_SEARCH}")

Aucune entité trouvée pour le train 9877
